# Computational Time Experiments

This notebook benchmarks FA and ZIFA runtime under varying cell counts and gene counts.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path(".")
DATA_DIR = Path("datasets/mouse_brain/datasets")
CORTEX_DIR = Path("datasets/cortex/datasets")
RESULT_DIR = Path("../results/scPI")
SAMPLED_DIR = RESULT_DIR / "sampled_data"
JOB_DIR = RESULT_DIR / "runtime_jobs"
LOG_DIR = RESULT_DIR / "logs"
FIGURE_DIR = RESULT_DIR / "figures"
RUNTIME_RESULTS_CSV = RESULT_DIR / "runtime_results.csv"

for path in [RESULT_DIR, SAMPLED_DIR, JOB_DIR, LOG_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_MOUSE_BRAIN = DATA_DIR / "1M_neurons_filtered_gene_bc_matrices_h5.h5"
PREPROCESS_META = SAMPLED_DIR / "mouse_brain_original_style_preprocess_meta.json"

SEED = 1234
K = 10
CELL_SIZES = [5000, 10000, 15000, 20000, 30000]
GENE_SIZES = [1000, 2000]

# Runtime settings
MAX_CPU_JOBS = 1
GPU_IDS = [1] 
OMP_NUM_THREADS_PER_CPU_JOB = "16"

# Match the original timing notebooks unless you intentionally tune these.
VI_THRESHOLD = 1e-2
VI_BATCH_SIZE = 128
VI_LR = 5e-4
VI_MAX_EPOCHS = 100000


## Raw Data Download Location

The mouse brain raw data is from 10x Genomics:

`https://cf.10xgenomics.com/samples/cell-exp/1.3.0/1M_neurons/1M_neurons_filtered_gene_bc_matrices_h5.h5`

Downloaded local path:

`datasets/mouse_brain/datasets/1M_neurons_filtered_gene_bc_matrices_h5.h5`

## Preprocess and Sample One Shared Dataset

This cell reads the raw 10x mouse brain h5 and follows the original preprocessing logic: for each cell scale, sample cells separately, compute gene standard deviations within that sampled subset, keep the top 2,000 genes, and cache the top 1,000/2,000 log1p matrices. For a fixed `(n_cells, n_genes)` setting, all methods use the exact same cached matrix.

In [2]:
def matrix_path(n_cells: int, n_genes: int) -> Path:
    return SAMPLED_DIR / f"mouse_brain_{n_cells//1000}k_{n_genes}g_log1p.npy"


def ensure_original_style_samples(force: bool = False) -> dict:
    expected = [matrix_path(n_cells, n_genes) for n_cells in CELL_SIZES for n_genes in GENE_SIZES]
    if all(p.exists() for p in expected) and PREPROCESS_META.exists() and not force:
        print("Using cached original-style sampled matrices in", SAMPLED_DIR)
        return json.loads(PREPROCESS_META.read_text())

    if not RAW_MOUSE_BRAIN.exists():
        raise FileNotFoundError(
            f"Raw mouse brain h5 is required for this notebook: {RAW_MOUSE_BRAIN}. "
            "Download it first, then rerun this cell."
        )

    import h5py
    from scipy import sparse

    rng = np.random.default_rng(SEED)
    max_genes = max(GENE_SIZES)
    records = []

    print(f"Reading raw 10x data without materializing the full matrix: {RAW_MOUSE_BRAIN}")
    with h5py.File(RAW_MOUSE_BRAIN, "r") as f:
        group_name = "matrix" if "matrix" in f else next(iter(f.keys()))
        g = f[group_name]
        shape = tuple(int(x) for x in g["shape"][:])
        n_genes_total, n_cells_total = shape
        if n_genes_total < max_genes:
            raise ValueError(f"Raw data has only {n_genes_total} genes, but {max_genes} are required.")

        source_indptr = g["indptr"]
        source_data = g["data"]
        source_indices = g["indices"]

        for n_cells in CELL_SIZES:
            if n_cells_total < n_cells:
                raise ValueError(f"Raw data has only {n_cells_total} cells, but {n_cells} are required.")

            print(f"\nSampling {n_cells:,} cells and selecting top genes by std...")
            cell_idx = rng.choice(n_cells_total, size=n_cells, replace=False)
            data_chunks = []
            index_chunks = []
            indptr = np.zeros(n_cells + 1, dtype=np.int64)

            for out_col, cell in enumerate(cell_idx):
                start = int(source_indptr[cell])
                end = int(source_indptr[cell + 1])
                data_chunks.append(source_data[start:end])
                index_chunks.append(source_indices[start:end])
                indptr[out_col + 1] = indptr[out_col] + (end - start)
                if (out_col + 1) % 5000 == 0 or (out_col + 1) == n_cells:
                    print(f"  loaded {out_col + 1:,}/{n_cells:,} cells; nnz={indptr[out_col + 1]:,}")

            data = np.concatenate(data_chunks).astype(np.float32, copy=False)
            indices = np.concatenate(index_chunks).astype(np.int32, copy=False)
            X_gene_by_cell = sparse.csc_matrix((data, indices, indptr), shape=(n_genes_total, n_cells))

            mean = np.asarray(X_gene_by_cell.mean(axis=1)).ravel()
            mean_sq = np.asarray(X_gene_by_cell.power(2).mean(axis=1)).ravel()
            gene_std = np.sqrt(np.maximum(mean_sq - mean ** 2, 0))
            top_gene_idx = np.argsort(gene_std)[::-1][:max_genes]
            X_2000 = X_gene_by_cell[top_gene_idx, :].T.toarray()
            X_2000 = np.log1p(X_2000).astype(np.float32, copy=False)

            np.save(SAMPLED_DIR / f"mouse_brain_{n_cells//1000}k_cell_idx.npy", cell_idx)
            np.save(SAMPLED_DIR / f"mouse_brain_{n_cells//1000}k_top2000_gene_idx.npy", top_gene_idx)

            for n_genes in GENE_SIZES:
                X = X_2000[:, :n_genes]
                out = matrix_path(n_cells, n_genes)
                np.save(out, X)
                print(f"  saved {out}, shape={X.shape}")
                records.append({
                    "n_cells": int(n_cells),
                    "n_genes": int(n_genes),
                    "matrix_path": str(out),
                })

    meta = {
        "source": str(RAW_MOUSE_BRAIN),
        "format": "10x_h5_old_mm10_group",
        "seed": SEED,
        "raw_shape_genes_by_cells": [int(n_genes_total), int(n_cells_total)],
        "cell_sizes": CELL_SIZES,
        "gene_sizes": GENE_SIZES,
        "transform": "log1p",
        "sampling": "Original-style preprocessing: each cell scale is sampled independently; genes are ranked by std within that scale.",
        "records": records,
    }
    PREPROCESS_META.write_text(json.dumps(meta, indent=2))
    print("Saved preprocess metadata:", PREPROCESS_META)
    return meta

preprocess_meta = ensure_original_style_samples(force=False)
print(pd.DataFrame(preprocess_meta["records"]))


Using cached original-style sampled matrices in ../results/scPI/sampled_data
   n_cells  n_genes                                        matrix_path
0     5000     1000  ../results/scPI/sampled_data/mouse_brain_5k_10...
1     5000     2000  ../results/scPI/sampled_data/mouse_brain_5k_20...
2    10000     1000  ../results/scPI/sampled_data/mouse_brain_10k_1...
3    10000     2000  ../results/scPI/sampled_data/mouse_brain_10k_2...
4    15000     1000  ../results/scPI/sampled_data/mouse_brain_15k_1...
5    15000     2000  ../results/scPI/sampled_data/mouse_brain_15k_2...
6    20000     1000  ../results/scPI/sampled_data/mouse_brain_20k_1...
7    20000     2000  ../results/scPI/sampled_data/mouse_brain_20k_2...
8    30000     1000  ../results/scPI/sampled_data/mouse_brain_30k_1...
9    30000     2000  ../results/scPI/sampled_data/mouse_brain_30k_2...


## Job Runner

The notebook writes a small local runner so each timing task executes in a fresh Python process. Each task loads the cached matrix for its `(n_cells, n_genes)` setting. This avoids CUDA state conflicts and lets CPU and GPU jobs run in parallel inside each experiment group.

In [3]:
RUNNER_PATH = RESULT_DIR / "runtime_job_runner.py"
RUNNER_PATH.write_text('\nfrom __future__ import annotations\n\nimport csv\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef _read_one_row_csv(path: Path) -> dict:\n    with path.open(newline="") as fh:\n        return next(csv.DictReader(fh))\n\n\ndef _write_one_row_csv(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=list(payload.keys()))\n        writer.writeheader()\n        writer.writerow(payload)\n\n\ndef main(job_csv: str) -> None:\n    spec_path = Path(job_csv)\n    spec = _read_one_row_csv(spec_path)\n\n    os.environ.setdefault("OMP_NUM_THREADS", str(spec.get("omp_num_threads", 1)))\n    os.environ.setdefault("MKL_NUM_THREADS", str(spec.get("omp_num_threads", 1)))\n    os.environ.setdefault("OPENBLAS_NUM_THREADS", str(spec.get("omp_num_threads", 1)))\n\n    import warnings\n    warnings.filterwarnings("ignore")\n\n    project_dir = spec.get("project_dir")\n    if project_dir:\n        sys.path.insert(0, project_dir)\n\n    X = np.asarray(np.load(spec["matrix_path"], mmap_mode="r"), dtype=np.float32)\n\n    result_path = Path(spec["result_path"])\n    result_path.parent.mkdir(parents=True, exist_ok=True)\n\n    start = time.time()\n    method = spec["method"]\n    K = int(spec["K"])\n    seed = int(spec.get("seed", 1234))\n    np.random.seed(seed)\n\n    if method == "FA":\n        from sklearn.decomposition import FactorAnalysis\n        model = FactorAnalysis(n_components=K, random_state=seed)\n        model.fit(X)\n        _ = model.transform(X)\n        extra = {"backend": "sklearn.FactorAnalysis", "device": "cpu"}\n\n    elif method in {"FA_VI", "FA_AVI"}:\n        import pyro\n        import torch\n        from FA import FA\n        pyro.clear_param_store()\n        torch.manual_seed(seed)\n        inference = "vi" if method == "FA_VI" else "amortized"\n        model = FA(\n            K=K,\n            method=inference,\n            threshold=float(spec["threshold"]),\n            batch_size=int(spec["batch_size"]),\n            lr=float(spec["lr"]),\n            max_epochs=int(spec["max_epochs"]),\n        )\n        model.fit(X)\n        extra = {"backend": "FA.py", "device": "cuda" if torch.cuda.is_available() else "cpu"}\n\n    elif method == "block_ZIFA":\n        from ZIFA import ZIFA\n        nonzero_cols = np.sum(X, axis=0) > 0\n        X_use = X[:, nonzero_cols]\n        model = ZIFA(K=K, method="block")\n        model.fit(X_use)\n        extra = {"backend": "ZIFA.py:block", "device": "cpu", "n_genes_after_filter": int(X_use.shape[1])}\n\n    elif method in {"ZIFA_VI", "ZIFA_AVI"}:\n        import pyro\n        import torch\n        from ZIFA import ZIFA\n        pyro.clear_param_store()\n        torch.manual_seed(seed)\n        inference = "vi" if method == "ZIFA_VI" else "amortized"\n        model = ZIFA(\n            K=K,\n            method="pyro",\n            inference=inference,\n            threshold=float(spec["threshold"]),\n            batch_size=int(spec["batch_size"]),\n            lr=float(spec["lr"]),\n            max_epochs=int(spec["max_epochs"]),\n        )\n        model.fit(X)\n        extra = {"backend": "ZIFA.py:pyro", "device": "cuda" if torch.cuda.is_available() else "cpu"}\n\n    else:\n        raise ValueError(f"Unknown method: {method}")\n\n    elapsed = time.time() - start\n    payload = {\n        "panel": spec["panel"],\n        "method": method,\n        "label": spec["label"],\n        "n_cells": int(spec["n_cells"]),\n        "n_genes": int(spec["n_genes"]),\n        "time_sec": elapsed,\n        "time_min": elapsed / 60.0,\n        "seed": seed,\n        "matrix_path": spec["matrix_path"],\n        **extra,\n    }\n    _write_one_row_csv(result_path, payload)\n    print(payload)\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n')
print("Wrote runner:", RUNNER_PATH)


Wrote runner: ../results/scPI/runtime_job_runner.py


## Runtime Helpers

In [4]:
def result_csv_path(panel: str, method: str, n_cells: int, n_genes: int) -> Path:
    return JOB_DIR / f"{panel}_{method}_{n_cells}_{n_genes}.csv"


def make_job(panel: str, n_genes: int, method: str, label: str, n_cells: int, resource: str) -> dict:
    result_path = result_csv_path(panel, method, n_cells, n_genes)
    return {
        "panel": panel,
        "method": method,
        "label": label,
        "resource": resource,
        "n_cells": n_cells,
        "n_genes": n_genes,
        "K": K,
        "seed": SEED,
        "matrix_path": str(matrix_path(n_cells, n_genes)),
        "project_dir": str(PROJECT_DIR),
        "result_path": str(result_path),
        "threshold": VI_THRESHOLD,
        "batch_size": VI_BATCH_SIZE,
        "lr": VI_LR,
        "max_epochs": VI_MAX_EPOCHS,
        "omp_num_threads": OMP_NUM_THREADS_PER_CPU_JOB,
    }


def collect_results() -> pd.DataFrame:
    frames = []
    for p in sorted(JOB_DIR.glob("*.csv")):
        if p.name.startswith("spec_"):
            continue
        try:
            frames.append(pd.read_csv(p))
        except pd.errors.EmptyDataError:
            print("Skipping incomplete csv:", p)
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not df.empty:
        for col in ["n_cells", "n_genes", "time_sec", "time_min", "seed"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.sort_values(["panel", "method", "n_genes", "n_cells"])
        df.to_csv(RUNTIME_RESULTS_CSV, index=False)
    return df


def run_group(panel: str, n_genes: int, methods: list[tuple[str, str, str]]) -> None:
    jobs = []
    for method, label, resource in methods:
        for n_cells in CELL_SIZES:
            jobs.append(make_job(panel, n_genes, method, label, n_cells, resource))

    pending = [job for job in jobs if not Path(job["result_path"]).exists()]
    skipped = len(jobs) - len(pending)
    print(f"{panel}: {len(pending)} pending, {skipped} already complete")
    if not pending:
        return

    running = []
    next_gpu = 0

    def launch(job: dict):
        nonlocal next_gpu
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        env["OMP_NUM_THREADS"] = str(job["omp_num_threads"])
        env["MKL_NUM_THREADS"] = str(job["omp_num_threads"])
        env["OPENBLAS_NUM_THREADS"] = str(job["omp_num_threads"])
        if job["resource"] == "gpu" and GPU_IDS:
            gpu = GPU_IDS[next_gpu % len(GPU_IDS)]
            next_gpu += 1
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
            slot = f"gpu{gpu}"
        elif job["resource"] == "gpu":
            env["CUDA_VISIBLE_DEVICES"] = ""
            slot = "cpu_forced_gpu_job"
        else:
            env["CUDA_VISIBLE_DEVICES"] = ""
            slot = "cpu"

        spec_path = JOB_DIR / f"spec_{job['panel']}_{job['method']}_{job['n_cells']}_{job['n_genes']}.csv"
        pd.DataFrame([job]).to_csv(spec_path, index=False)
        log_path = LOG_DIR / f"{job['panel']}_{job['method']}_{job['n_cells']}_{job['n_genes']}.log"
        log_fh = open(log_path, "w")
        proc = subprocess.Popen(
            [sys.executable, str(RUNNER_PATH), str(spec_path)],
            cwd=str(PROJECT_DIR),
            env=env,
            stdout=log_fh,
            stderr=subprocess.STDOUT,
        )
        print(f"Launched {job['panel']} {job['method']} {job['n_cells']} cells {job['n_genes']} genes on {slot}; log={log_path}")
        return {"proc": proc, "job": job, "log_fh": log_fh, "slot": slot}

    while pending or running:
        cpu_running = sum(1 for r in running if r["job"]["resource"] == "cpu")
        gpu_running = sum(1 for r in running if r["job"]["resource"] == "gpu")
        gpu_capacity = max(1, len(GPU_IDS)) if GPU_IDS else 1

        for job in list(pending):
            if job["resource"] == "cpu" and cpu_running < MAX_CPU_JOBS:
                running.append(launch(job))
                pending.remove(job)
                cpu_running += 1
            elif job["resource"] == "gpu" and gpu_running < gpu_capacity:
                running.append(launch(job))
                pending.remove(job)
                gpu_running += 1

        time.sleep(10 if running else 1)

        for r in list(running):
            rc = r["proc"].poll()
            if rc is None:
                continue
            r["log_fh"].close()
            running.remove(r)
            job = r["job"]
            if rc != 0:
                failed_log = LOG_DIR / f"{job['panel']}_{job['method']}_{job['n_cells']}_{job['n_genes']}.log"
                raise RuntimeError(
                    f"Job failed rc={rc}: {job['panel']} {job['method']} {job['n_cells']} {job['n_genes']}. "
                    f"See {failed_log}"
                )
            print(f"Finished {job['panel']} {job['method']} {job['n_cells']} cells")

    df = collect_results()
    print(df.tail())


## Run Four Experiment Groups

The groups are intentionally run in this order: FA 1,000 genes, FA 2,000 genes, ZIFA 1,000 genes, ZIFA 2,000 genes.

In [ ]:
FA_METHODS = [
    ("FA", "FA", "cpu"),
    ("FA_VI", "VI", "gpu"),
    ("FA_AVI", "AVI", "gpu"),
]

ZIFA_METHODS = [
    ("block_ZIFA", "block_ZIFA", "cpu"),
    ("ZIFA_VI", "VI", "gpu"),
    ("ZIFA_AVI", "AVI", "gpu"),
]

run_group("FA_1000", 1000, FA_METHODS)
run_group("FA_2000", 2000, FA_METHODS)
run_group("ZIFA_1000", 1000, ZIFA_METHODS)
run_group("ZIFA_2000", 2000, ZIFA_METHODS)

runtime_df = collect_results()
runtime_df

FA_1000: 7 pending, 8 already complete
Launched FA_1000 FA_VI 20000 cells 1000 genes on gpu0; log=../results/scPI/logs/FA_1000_FA_VI_20000_1000.log


## Plot Runtime Figure

Matplotlib default color cycle is used intentionally.

In [ ]:
runtime_df = pd.read_csv(RUNTIME_RESULTS_CSV) if RUNTIME_RESULTS_CSV.exists() else collect_results()
if runtime_df.empty:
    raise ValueError("No runtime results found yet.")

panel_specs = [
    ("FA_1000", "FA - 1,000 genes", ["FA", "VI", "AVI"]),
    ("ZIFA_1000", "ZIFA - 1,000 genes", ["block_ZIFA", "VI", "AVI"]),
    ("FA_2000", "FA - 2,000 genes", ["FA", "VI", "AVI"]),
    ("ZIFA_2000", "ZIFA - 2,000 genes", ["block_ZIFA", "VI", "AVI"]),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), sharey=False)
for ax, (panel, title, labels) in zip(axes, panel_specs):
    sub = runtime_df[runtime_df["panel"] == panel]
    for label in labels:
        s = sub[sub["label"] == label].sort_values("n_cells")
        ax.plot(s["n_cells"] / 1000, s["time_min"], marker="o", label=label)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Number of cells (thousands)", fontweight="bold")
    ax.set_ylabel("Computational time (min)", fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=True)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "computational_time_all_panels.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURE_DIR / "computational_time_all_panels.pdf", bbox_inches="tight")
plt.show()


In [ ]:
runtime_df = pd.read_csv(RUNTIME_RESULTS_CSV) if RUNTIME_RESULTS_CSV.exists() else collect_results()
for panel, title, labels in panel_specs:
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    sub = runtime_df[runtime_df["panel"] == panel]
    for label in labels:
        s = sub[sub["label"] == label].sort_values("n_cells")
        ax.plot(s["n_cells"] / 1000, s["time_min"], marker="o", label=label)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Number of cells (thousands)", fontweight="bold")
    ax.set_ylabel("Computational time (min)", fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=True)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"computational_time_{panel}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"computational_time_{panel}.pdf", bbox_inches="tight")
    plt.show()
